# Scratch — Monday Step 1: Query + Load Check

Quick, disposable checks against the real local MongoDB. Not part of the final wrangle() build — just verifying the raw query works and the data shape is what we expect before writing cleaning logic.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from pymongo import MongoClient

client = MongoClient(host="localhost", port=27017)
db = client["air-quality"]
dar = db["dar-es-salaam"]

# sanity check: confirm the collection is actually seeded
print("Total documents in collection:", dar.count_documents({}))


## Step 1: query site 11, P2 readings only

In [ ]:
results = dar.find(
    {"metadata.site": 11, "metadata.measurement": "P2"},
    projection={"P2": 1, "timestamp": 1, "_id": 0},
)
df = pd.DataFrame(list(results)).set_index("timestamp")
df.shape


In [ ]:
df.head()


## Checks requested before moving to Step 2

1. Index dtype — is it already a proper datetime, or something else (string, object)?
2. Is the index tz-naive or tz-aware?
3. Any obvious garbage (nulls, wrong dtypes on P2)?

In [ ]:
print("Index dtype:", df.index.dtype)
print("Index tz:", getattr(df.index, "tz", "N/A (not a DatetimeIndex if this errors)"))
print()
print(df.dtypes)
print()
print("Nulls in P2:", df["P2"].isna().sum())
print("P2 min/max:", df["P2"].min(), df["P2"].max())


In [ ]:
# If the index came back as plain object/string rather than a real
# DatetimeIndex, this is where you would find out and fix it before
# attempting tz_localize() in Step 2, e.g.:
# df.index = pd.to_datetime(df.index)
type(df.index)
